In [1]:
%load_ext autoreload
%autoreload 2


# Collecting data for uncovered transactions

As a result of the script's execution, a new database file uncovered_transactions will be created. The database contains all transactions that are not covered by existing patterns. Only the verbs present in the pattern database are considered covered.

Tables created in this script:

* transaction_head
* transaction_row
* verbs_table
* verb_transactions
* patterns

The inputs are:

* verb pattern database
* verb transaction database



In [2]:
import sys
import sqlite3
from sqlalchemy import create_engine
sys.path.append('../../../common_code')
from paths import PATH_ROOT
from db_operations.db_table_ops import copy_table_structure
from db_operations.verb_transactions.filter_verb_transaction_tables import filter_verb_transaction_tables
from db_operations.db_display import display_sqlite_as_dataframe

from helpers import helpers

### Configuration

In [3]:
DB_DIR = "./example_data"
PATH_TRANSACTIONS_DB = DB_DIR + "/transactions.db"
PATH_PATTERNS_DB = DB_DIR + "/patterns.db"
UNCOVERED_TRANSACTIONS_DB = DB_DIR + "/uncovered_transactions.db"

PATTERNS_SOURCE_SCHEMA = "db_pat"
TRX_SOURCE_SCHEMA = 'db_tr'

### Connect to db

In [4]:
# tekitab andmebaasifaili, kui seda veel ei olnud
UNCOVERED_TRANSACTIONS_DB_PATH = f"sqlite:///{UNCOVERED_TRANSACTIONS_DB}"
con = sqlite3.connect(UNCOVERED_TRANSACTIONS_DB)
con.close()

engine = create_engine(UNCOVERED_TRANSACTIONS_DB_PATH)
helpers.reset_tables(engine)
conn = engine.connect()
conn.close()

# use sqlalchemy for easier tbl creation
conn = sqlite3.connect(UNCOVERED_TRANSACTIONS_DB)

conn.row_factory = sqlite3.Row

# liidame teised andmebaasid
conn.execute(f"ATTACH DATABASE '{PATH_PATTERNS_DB}' AS {PATTERNS_SOURCE_SCHEMA}")
conn.execute(f"ATTACH DATABASE '{PATH_TRANSACTIONS_DB}' AS {TRX_SOURCE_SCHEMA}")

### Workflow

Loome tabeli **verbs**.

#### I verbs
| väli          | tüüp | kirjeldus                               | näide    | märkus          |
| ------------- | ---- | --------------------------------------- | -------- | --------------- |
| verb_id       | int  | verbi unikaalne id                      |          |                 |
| verb          | text | verbi lemma                             | _aasima_ |                 |
| verb_compound | text | verbi afiksaaladverbid                  |          | eraldajaks koma |
| pat_ids       | text | mustrite id-d verb_patterns andmebaasis |          | eraldajaks koma |


In [5]:
%%time
# täidame verbs tabeli
helpers.fill_table_verbs(conn=conn)
verbs = conn.execute(
    "SELECT verb_id, pat_ids, verb, verb_compound FROM verbs;"
    ).fetchall()
verbs = [dict(row) for row in verbs]
print(f"Andmebaasi lisati {len(verbs)} verbi")

Andmebaasi lisati 2 verbi
CPU times: user 590 μs, sys: 1.76 ms, total: 2.35 ms
Wall time: 1.98 ms


In [6]:
display_sqlite_as_dataframe(db_path=UNCOVERED_TRANSACTIONS_DB, table_name='verbs', n_rows=5)

,verb_id,verb,verb_compound,pat_ids
0,1,aasima,,"1,2"
1,2,neelama,alla,32


Kopeerime verbi mustrite baasist tabeli **patterns** (sisu ja struktuuri).

#### II patterns (koopia mustrite baasist)

| väli          | tüüp | kirjeldus | näide | märkus |
| ------------- | ---- | --------- | ----- | ------ |
| pat_id        | int  |           |       |        |
| pattern       | text |           |       |        |
| verb_word     | text |           |       |        |
| verb_compound | text |           |       |        |
| phrase_nr     | int  |           |       |        |
| phrase_case   | text |           |       |        |
| adp           | text |           |       |        |
| inf_verb      | text |           |       |        |

In [7]:
%%time
# kopeerime mustride tabeli andmetega
copy_table_structure(conn, table_name='db_pat.patterns', new_table_name='patterns', copy_data=True, delete_if_exists=True, verbose=False)

CPU times: user 477 μs, sys: 816 μs, total: 1.29 ms
Wall time: 926 μs


('main', 'patterns')

In [8]:
display_sqlite_as_dataframe(db_path=UNCOVERED_TRANSACTIONS_DB, table_name='patterns', n_rows=5)

,pat_id,pattern,verb_word,verb_compound,phrase_nr,phrase_case,adp,inf_verb
0,1,aasima keda*,aasima,,1,part,,
1,2,aasima kelle kallal,aasima,,1,gen,kallal,
2,32,alla neelama mida,neelama,alla,1,part,,


Täidame **verb_transactions** tabeli sisuga.

#### III verb_transactions

| väli    | tüüp | kirjeldus         | näide | märkus |
| ------- | ---- | ----------------- | ----- | ------ |
| verb_id | int  | verbi id          |       |        |
| head_id | int  | transaktsiooni id |       |        |



In [9]:
%%time

from tqdm import tqdm
for v in tqdm(verbs[:2]):
    helpers.fill_table_verb_transactions(conn=conn, verb_id=v['verb_id'], pat_ids=v['pat_ids'].split(','))
conn.execute("SELECT COUNT(verb_id) FROM verb_transactions").fetchone()[0]

100%|██████████| 2/2 [00:00<00:00, 916.69it/s]

CPU times: user 6.02 ms, sys: 5.94 ms, total: 12 ms
Wall time: 14.9 ms


692

In [10]:
display_sqlite_as_dataframe(db_path=UNCOVERED_TRANSACTIONS_DB, table_name='verb_transactions', n_rows=5)

,verb_id,head_id
0,1,52151
1,1,533958
2,1,603867
3,1,663669
4,1,750763


Tekitame **transaction_head** ja **transaction_row** tabelid, milles sisalduvad ainult need transaktsioonid, mis ei olnud kaetud mustriga. 
 (võtab ca paarkümmend minutit aega).

####  IV transaction_head

| väli          | tüüp | kirjeldus                                                                         | näide      | märkus          |
| ------------- | ---- | --------------------------------------------------------------------------------- | ---------- | --------------- |
| id            | int  | rea <br/>unikaalne ID                                                             | _56_       |                 |
| sentence_id   | int  | lause id andmebaasis                                                              |            |                 |
| loc           | int  | verbi asukoht lauses                                                              |            |                 |
| verb          | text | verbi lemma                                                                       | _olema_    |                 |
| verb_compound | text | verbi afiksaaladverbid                                                            | alla,peale | eraldajaks koma |
| form          | text | verb sellises vormis, nagu see lauses esines                                      | _oli_      |                 |
| deprel        | text | verbi deprel                                                                      |            |                 |
| feats         | text | verbi morf kategooriad alfabeetilises järjekorras                                 | aux,ps3    |                 |
| phrase        | text | puhastatud fraas (ainult need alluvad, mis on transactions tabelisse salvestatud) |            |                 |

#### V transaction_row

| väli       | tüüp | kirjeldus                                                            | näide  | märkus |
| ---------- | ---- | -------------------------------------------------------------------- | ------ | ------ |
| id         | int  | rea <br/>unikaalne ID                                                | _56_   |        |
| head_id    | int  | rea transaction_head.id                                              |        |        |
| loc        | int  | sõna asukoht lauses                                                  |        |        |
| loc_rel    | int  | sõna asukoht verbi suhtes                                            |        |        |
| deprel     | text | sõna deprel                                                          |        |        |
| form       | text | sõna vorm                                                            |        |        |
| lemma      | text | sõna lemma                                                           |        |        |
| pos        | text | sõna sõnaliik                                                        |        |        |
| feats      | text | sõna morf kategooriad alfabeetilises järjekorras                     | add,sg |        |
| parent_loc | int  | vanema tipu loc, juhul kui tegemist on <code>obl</obl> alluvaga case | 2      |        |

In [11]:
%%time
# Kopeeritakse tabelite struktuur db_tr andmebaasist
# Täidetakse sisuga, tehakse join verb_transactions.head_id tabeliga

filter_verb_transaction_tables(
    conn=conn,
    source_schema=TRX_SOURCE_SCHEMA,
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    ids_table='verb_transactions',
    ids_column='head_id',
    delete_if_exists=True,
    copy_indexes=True,
    verbose=False,
)


CPU times: user 10.6 ms, sys: 6.86 ms, total: 17.4 ms
Wall time: 25 ms


,Parameter,Schema,Table,Rows,Time (sec)
0,head_ids unique values,main,verb_transactions,692,---
1,head_ids_table,main,verb_transactions,692,---
2,transaction_head,db_tr,transaction_head,914,---
3,transaction_row,db_tr,transaction_row,3108,---
4,new_transaction_head,main,transaction_head,692,0.005
5,new_transaction_row,main,transaction_row,2205,0.005
6,fetching rows count,---,---,---,0.001
7,creating tables,---,---,---,0.012
8,committing result to db,---,---,---,0.001
9,total time,---,---,---,0.024


In [12]:
#%%time
# kontrollime 10 juhusliku verbi pealt, et numbrid jooksevad kokku
if 1:
    import random
    random_i = random.sample(range(0, len(verbs)-1), min(10, len(verbs)-1))
    # check verbs stat
    # get counts of random transactions to check, that numbers align together
    for v in [verbs[i] for i in random_i]:
        helpers.show_verb_trans_stat(conn=conn, verb=v, trx_source_schema=TRX_SOURCE_SCHEMA, patterns_src_schema=PATTERNS_SOURCE_SCHEMA)
    

verb {'verb_id': 1, 'pat_ids': '1,2', 'verb': 'aasima', 'verb_compound': ''}
db_pat total: 176
db_pat unmatched: 119
db_pat matched: 57
db_tr all: 176
uncovered_transactions all: 119
 


In [13]:
conn.close()